# ReAct Agent with LangGraph

A ReAct (Reasoning + Acting) agent built with LangGraph. The model reasons about a question, calls tools when it needs them, reads the results, and repeats until it can answer.

Tools:
Tools:
- `search_tool`: web search through Tavily for CVEs, threat intel, and adversary TTPs
- `analyze_ioc`: triage analyzer for IP addresses, hashes, domains, and suspicious ports

Set `OPENAI_API_KEY` and `TAVILY_API_KEY` as environment variables before running.

In [ ]:
%pip install -q langgraph==0.3.34 langchain==0.3.24 langchain-openai==0.3.14 langchain-community==0.3.23

## Tools

In [ ]:
import ipaddress
import json
import re
import warnings
warnings.filterwarnings('ignore')

from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_core.tools import tool

search = TavilySearchResults()  # reads TAVILY_API_KEY from the environment


@tool
def search_tool(query: str):
    """
    Search the web for information using Tavily API.
    Useful for looking up CVE details, threat actor profiles, recent breach reports, or unfamiliar malware families.

    :param query: The search query string
    :return: Search results related to the query
    """
    return search.invoke(query)


@tool
def analyze_ioc(indicator: str) -> str:
    """
    Analyze a security indicator (IP address, file hash, domain, or port) 
    and provide initial triage assessment and recommended defensive actions.

    :param indicator: An IP address, MD5/SHA256 hash, domain name, or port number.
    :return: A triage assessment with recommended containment/investigation steps.
    """
    indicator = indicator.strip()
    
    # 1. Port analysis
    if indicator.isdigit():
        port = int(indicator)
        high_risk_ports = {
            22: "SSH - Check for external exposure and brute-force attempts.",
            23: "Telnet - Insecure cleartext protocol; isolate and replace immediately.",
            445: "SMB - High-risk vector for lateral movement, ransomware, and EternalBlue exploits.",
            3389: "RDP - Common attack vector; verify MFA, external accessibility, and failed logon spikes.",
            4444: "Metasploit default listener - Strong indicator of active reverse shell/compromise.",
            8080: "HTTP Alternate - Inspect web application logs and proxy traffic for suspicious activity."
        }
        advice = high_risk_ports.get(port, "Standard or non-critical port. Inspect associated process and outbound connections.")
        return f"[Port {port} Triage]: {advice}"

    # 2. File Hash analysis (MD5 / SHA256)
    if re.fullmatch(r"[a-fA-F0-9]{32}", indicator):
        return (
            f"[MD5 Hash: {indicator}]: Standard MD5 hash detected. "
            "Recommendation: Query hash against threat intelligence feeds (e.g., VirusTotal), check EDR process execution logs, "
            "and compute SHA256 for definitive sample tracking."
        )
    if re.fullmatch(r"[a-fA-F0-9]{64}", indicator):
        return (
            f"[SHA256 Hash: {indicator}]: SHA256 cryptographic hash detected. "
            "Recommendation: Search EDR telemetry for child processes, spawned command lines, persistence mechanisms, "
            "and cross-reference search_tool for associated malware families or APT campaigns."
        )

    # 3. IP Address analysis
    try:
        ip = ipaddress.ip_address(indicator)
        if ip.is_private:
            return (
                f"[Internal RFC-1918 IP: {indicator}]: Private address. "
                "Recommendation: Check internal network flows for lateral movement, abnormal SMB/RPC traffic, "
                "and inspect Windows event logs (Event ID 4624/4625) on source and destination hosts."
            )
        if ip.is_loopback:
            return f"[Loopback IP: {indicator}]: Localhost address. Verify process binding and local service configuration."
        return (
            f"[Public IP: {indicator}]: External routable IP address. "
            "Recommendation: Use search_tool to check ASN reputation and threat intelligence. "
            "Check perimeter firewall/proxy logs for outbound C2 beaconing or large data egress."
        )
    except ValueError:
        pass

    # 4. Domain Name heuristic
    if "." in indicator and not indicator.startswith("http"):
        return (
            f"[Domain/FQDN: {indicator}]: Inspect DNS request logs for DGAs (Domain Generation Algorithms), "
            "evaluate domain age via WHOIS search, and review TLS certificates and web proxy access logs."
        )

    return (
        f"[Unclassified Indicator: {indicator}]: Verify the indicator format (must be IP, hash, port, or domain) "
        "or use search_tool to query broader threat intelligence."
    )


tools = [search_tool, analyze_ioc]
tools_by_name = {t.name: t for t in tools}

## Model and prompt

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, ToolMessage

model = ChatOpenAI(model='gpt-4o-mini')

chat_prompt = ChatPromptTemplate.from_messages([
    ('system', """
You are an expert Cybersecurity Incident Response & Threat Hunting AI Assistant.

When analyzing security alerts, incidents, or suspicious indicators:
1. Identify all indicators of compromise (IoCs), ports, domains, or hashes mentioned.
2. Use `analyze_ioc` to get initial triage assessments and containment recommendations.
3. Use `search_tool` to look up recent CVE advisories, malware families, threat actor TTPs, or IP/domain reputation.
4. Synthesize your technical findings into clear, structured incident response recommendations (Containment, Eradication, and Lessons Learned).
"""),
    MessagesPlaceholder(variable_name='scratch_pad'),
])

model_react = chat_prompt | model.bind_tools(tools)

## Agent state and graph

The state is the running message history. The graph loops between the `agent` node (the model) and the `tools` node until the model stops asking for tools.

In [ ]:
from typing import Annotated, Sequence, TypedDict
from langchain_core.messages import BaseMessage
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages


class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]


def call_model(state: AgentState):
    """Run the model on the conversation so far."""
    response = model_react.invoke({'scratch_pad': state['messages']})
    return {'messages': [response]}


def tool_node(state: AgentState):
    """Execute every tool call requested in the last message."""
    outputs = []
    for tool_call in state['messages'][-1].tool_calls:
        result = tools_by_name[tool_call['name']].invoke(tool_call['args'])
        outputs.append(
            ToolMessage(
                content=json.dumps(result),
                name=tool_call['name'],
                tool_call_id=tool_call['id'],
            )
        )
    return {'messages': outputs}


def should_continue(state: AgentState):
    """Go to the tools node if the model asked for tools, otherwise finish."""
    if state['messages'][-1].tool_calls:
        return 'continue'
    return 'end'


workflow = StateGraph(AgentState)
workflow.add_node('agent', call_model)
workflow.add_node('tools', tool_node)

workflow.set_entry_point('agent')
workflow.add_conditional_edges('agent', should_continue, {'continue': 'tools', 'end': END})
workflow.add_edge('tools', 'agent')

graph = workflow.compile()

In [ ]:
from IPython.display import Image, display

try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    pass  # needs extra dependencies; the diagram is optional

## Run the agent

In [ ]:
def print_stream(stream):
    for s in stream:
        message = s['messages'][-1]
        if isinstance(message, tuple):
            print(message)
        else:
            message.pretty_print()


inputs = {
    'messages': [
        HumanMessage(
            content=(
                "We detected suspicious outbound traffic to 185.220.101.5 on port 4444, "
                "and found an executable with SHA256 5e884898da28047151d0e56f8dc6292773603d0d6aabbdd62a11ef721d1542d8. "
                "What is going on here and what should our incident response team do immediately?"
            )
        )
    ]
}
print_stream(graph.stream(inputs, stream_mode='values'))